In [4]:
library(readxl)
library(readr)
library(dplyr)
library(ggplot2)
library(tidyr)
library(ggrepel)
library(networkD3)
library(tibble)
library(ggalluvial)
library(RColorBrewer)
library(scales)
library(viridis)


Attaching package: ‘readr’


The following object is masked from ‘package:scales’:

    col_factor




In [1]:
load("/vol/projects/yzhang/500FG_aging/output/02_Olink/olink_gender.RData")

In [2]:
olink <- readRDS("/vol/projects/CIIM/cohorts_old/500FG/olinkData/NPX_FG500.RDS")
head(olink)
dim(olink)
any(is.na(olink))

,IL8,VEGFA,CD8A,CDCP1,CD244,IL7,OPG,LAP_TGF_beta_1,uPA,IL6,⋯,CX3CL1,TNFRSF9,NT_3,TWEAK,CCL20,ST1A1,STAMBP,ADA,TNFB,CSF_1
HV001,3.63611,9.48365,9.52098,2.30522,7.29957,2.53580,9.68034,6.69739,9.83187,2.65429,⋯,5.09046,6.45090,2.70225,9.03842,8.68959,3.22945,4.78947,4.93394,4.44519,9.53556
HV002,4.31364,9.69720,10.49119,1.97309,7.34675,3.16577,10.23258,7.63274,10.44883,2.01410,⋯,5.44641,6.73492,2.76555,9.75981,6.22762,4.50936,5.30881,5.22062,4.81664,9.72443
HV003,4.06539,10.19810,9.57446,2.28619,7.46174,2.91413,10.33496,7.45751,9.80417,3.37359,⋯,6.08994,6.77356,2.77692,8.77353,6.15381,4.79457,5.47104,5.15920,5.64723,10.37471
HV004,3.20146,9.97967,7.86780,2.22472,6.33368,2.29093,9.83527,6.53851,9.56768,3.32486,⋯,5.59638,6.68823,2.44064,8.87093,5.66952,4.72120,4.61117,5.74346,5.96514,10.25552
HV005,4.28828,10.10370,11.63828,2.26651,7.93486,3.32575,9.80797,7.58017,10.16342,2.64213,⋯,6.07405,7.26541,2.69546,9.52142,7.18970,4.65843,5.87574,5.37253,5.17238,9.85607
HV006,4.21672,10.11105,11.17692,2.53762,7.29801,2.96055,10.59224,6.94480,10.04118,2.97745,⋯,6.00473,7.15083,2.69737,9.67547,6.45433,3.70019,4.68181,5.17577,5.08267,9.98597


[1] 424  70

[1] FALSE

In [16]:
basicPhenos = read.csv("/vol/projects/yzhang/500FG_aging/output/02_Olink/Age_group_basicPhenos.csv",row.names=1)
rownames(basicPhenos) = basicPhenos$ID_500fg
# Get shared sample IDs between olink and phenotype table
idx <- intersect(rownames(olink), basicPhenos$ID_500fg)
length(idx)  # 458
# Subset olink to shared IDs (keeps olink row order)
olink_filter <- olink[rownames(olink) %in% idx, , drop = FALSE]
head(olink_filter)
dim(olink_filter)
# Subset phenotypes and reorder to exactly match olink_filter rownames
basicPhenos_filter_match <- basicPhenos %>%
  filter(ID_500fg %in% rownames(olink_filter)) %>%
  slice(match(rownames(olink_filter), ID_500fg))
# Sanity check: IDs should align 1-to-1
sum(rownames(olink_filter) == basicPhenos_filter_match$ID_500fg)
head(basicPhenos_filter_match)
dim(basicPhenos_filter_match)  # 458 x (pheno columns)

##one missing gender value
basicPhenos_filter_match$Gender <- trimws(basicPhenos_filter_match$Gender)
basicPhenos_filter_match$Gender[basicPhenos_filter_match$Gender == ""] <- NA

[1] 417

,IL8,VEGFA,CD8A,CDCP1,CD244,IL7,OPG,LAP_TGF_beta_1,uPA,IL6,⋯,CX3CL1,TNFRSF9,NT_3,TWEAK,CCL20,ST1A1,STAMBP,ADA,TNFB,CSF_1
HV001,3.63611,9.48365,9.52098,2.30522,7.29957,2.53580,9.68034,6.69739,9.83187,2.65429,⋯,5.09046,6.45090,2.70225,9.03842,8.68959,3.22945,4.78947,4.93394,4.44519,9.53556
HV002,4.31364,9.69720,10.49119,1.97309,7.34675,3.16577,10.23258,7.63274,10.44883,2.01410,⋯,5.44641,6.73492,2.76555,9.75981,6.22762,4.50936,5.30881,5.22062,4.81664,9.72443
HV003,4.06539,10.19810,9.57446,2.28619,7.46174,2.91413,10.33496,7.45751,9.80417,3.37359,⋯,6.08994,6.77356,2.77692,8.77353,6.15381,4.79457,5.47104,5.15920,5.64723,10.37471
HV004,3.20146,9.97967,7.86780,2.22472,6.33368,2.29093,9.83527,6.53851,9.56768,3.32486,⋯,5.59638,6.68823,2.44064,8.87093,5.66952,4.72120,4.61117,5.74346,5.96514,10.25552
HV005,4.28828,10.10370,11.63828,2.26651,7.93486,3.32575,9.80797,7.58017,10.16342,2.64213,⋯,6.07405,7.26541,2.69546,9.52142,7.18970,4.65843,5.87574,5.37253,5.17238,9.85607
HV006,4.21672,10.11105,11.17692,2.53762,7.29801,2.96055,10.59224,6.94480,10.04118,2.97745,⋯,6.00473,7.15083,2.69737,9.67547,6.45433,3.70019,4.68181,5.17577,5.08267,9.98597


[1] 417  70

[1] 417

,ID_500fg,Gender,Age,Height,Weight,BMI,LivingSituation,Education,Ethnicity,FarmAnimal,Pets,Travel,DayInMonth,monthinYear,dayInYear,year,AgeGroup
,<chr>,<chr>,<int>,<int>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<chr>
HV001,HV001,male,27,178,70,22.09317,Married,University degree,European,Once a month,Yes,No,24,7,204,2013,Group 5
HV002,HV002,male,26,178,68,21.46194,Cohabitation,University degree,European,Rarely,No,Yes,24,7,204,2013,Group 4
HV003,HV003,female,24,173,68,22.72044,Married,University degree,European,Rarely,Yes,Yes,26,8,236,2013,Group 4
HV004,HV004,female,20,173,65,21.71807,Married,University degree,European,Rarely,Yes,No,26,8,236,2013,Group 2
HV005,HV005,female,22,178,64,20.19947,Married,University student,European,Once a month,Yes,Yes,5,9,245,2013,Group 3
HV006,HV006,female,24,168,58,20.54989,Married,University degree,European,Rarely,Yes,No,NA,NA,NA,NA,Group 4


[1] 417  17

In [8]:
table(basicPhenos$Gender)
idx <- which(trimws(as.character(basicPhenos$Gender)) == "")
basicPhenos[idx, ] 


       female   male 
     1    293    233 

,ID_500fg,Gender,Age,Height,Weight,BMI,LivingSituation,Education,Ethnicity,FarmAnimal,Pets,Travel,DayInMonth,monthinYear,dayInYear,year,AgeGroup
,<chr>,<chr>,<int>,<int>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<chr>
HV322,HV322,,67,158,52,20.83,Married,Hogeschool opleiding (HBO),European,Rarely,No,No,16,6,166,2014,Group 6


In [19]:
# model 1: age + protein (age not scaled) + Gender
all_results <- list()

for (i in seq_len(ncol(olink_filter))) {

  # Build a minimal modeling table
  data <- data.frame(
    feature = log(olink_filter[, i] + 1),
    age = basicPhenos_filter_match$Age,
    Gender = factor(basicPhenos_filter_match$Gender)  # convert chr to factor
  )

  # Linear regression
  mod <- lm(feature ~ age + Gender, data = data)

  # Extract coefficient and p-value for age (not Gender)
  cf <- summary(mod)$coefficients
  res <- cf["age", c("Estimate", "Pr(>|t|)")]

  # Store results
  result <- data.frame(
    feature  = colnames(olink_filter)[i],
    estimate = res["Estimate"],
    p        = res["Pr(>|t|)"]
  )

  # Signed -log10(p)
  result$value <- -log10(result$p) * result$estimate

  all_results[[i]] <- result
}

final_result <- do.call(rbind, all_results)
final_result$padj <- p.adjust(final_result$p, method = "BH")
final_result$sig <- ifelse(final_result$padj < 0.05, "sig", "no")

head(final_result)
dim(final_result)
table(final_result$sig)

write.csv(final_result,
          file = "/vol/projects/yzhang/500FG_aging/output/02_Olink/olink_age_liner_gender_FDR.csv")

,feature,estimate,p,value,padj,sig
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
Estimate,IL8,1.857226e-03,1.605897e-07,1.261852e-02,1.405159e-06,sig
Estimate1,VEGFA,3.326290e-04,2.074781e-03,8.924529e-04,6.051445e-03,sig
Estimate2,CD8A,-1.732231e-03,1.947852e-06,-9.891808e-03,1.136247e-05,sig
Estimate3,CDCP1,8.233146e-03,1.164648e-42,3.452471e-01,8.152538e-41,sig
Estimate4,CD244,-1.948333e-05,9.125587e-01,-7.742512e-07,9.257842e-01,no
Estimate5,IL7,1.341586e-03,3.960574e-02,1.881228e-03,7.108723e-02,no


[1] 70  6


 no sig 
 33  37 

In [14]:
# 添加列：显著性分类（上调/下调/非显著）
final_result <- final_result %>% 
  mutate(
    sig = ifelse(p < 0.05 & estimate > 0, "upregulated",
                 ifelse(p < 0.05 & estimate < 0, "downregulated", "not significant")),
    log_p = -log10(p) 
  )

# 分别筛选上调和下调中 log_p 最大的前 4 个
top3_upregulated <- final_result %>%
  filter(sig == "upregulated") %>%
  arrange(desc(log_p)) %>%
  slice(1:3)

top3_downregulated <- final_result %>%
  filter(sig == "downregulated") %>%
  arrange(desc(log_p)) %>%
  slice(1:3)

# 合并上调和下调的前 3 个显著点
top4_combined <- bind_rows(top3_upregulated, top3_downregulated)

# 设置保存火山图的分辨率和尺寸
ppi <- 300
png("/vol/projects/yzhang/500FG_aging/output/02_Olink/volcano_age_olink_liner_top3(2).png", 
    width = 5 * ppi, height = 4 * ppi, res = ppi)

# 绘制火山图
ggplot(final_result, aes(x = estimate, y = log_p)) + 
  geom_point(aes(color = sig), size = 3, alpha = 0.6) +  # 显著性颜色
  scale_color_manual(values = c(
    "upregulated" = "#FFB3E6", 
    "downregulated" = "#80B1D3", 
    "not significant" = "grey"
  )) + 
  theme_classic() + 
  xlab("Estimate (Effect Size)") +
  ylab("-log10(p)") +
  ggtitle("Proteomics ~ Age Volcano Plot") +
  # 显示显著性统计信息
  annotate(
    "text", 
    x = max(final_result$estimate, na.rm = TRUE) * -1.3, 
    y = max(final_result$log_p, na.rm = TRUE) - 3, 
    label = paste0(
     "Upregulated: ", sum(final_result$sig == "upregulated"), 
     " (", round(mean(final_result$sig == "upregulated") * 100, 2), "%)", 
     "\nDownregulated: ", sum(final_result$sig == "downregulated"), 
     " (", round(mean(final_result$sig == "downregulated") * 100, 2), "%)"
    ), 
    size = 4, 
    hjust = 0, 
    color = "black"
  ) +
  # 添加 top 4 显著点的名字，使用 geom_text_repel 避免重叠
  geom_text_repel(
    data = top4_combined,
    aes(x = estimate, y = log_p, label = olink),
    size = 3.5, 
    color = "black",
    box.padding = 0.5,  # 控制文本与点的距离
    point.padding = 0.5,  # 控制点与文本的距离
    #max.overlaps = Inf  # 避免重叠
  )

dev.off()

png 
  2

In [13]:
table(final_result$sig)
protein_sig <- final_result[final_result$sig=="sig",]
dim(protein_sig)


 no sig 
 33  37 

[1] 37  6

In [9]:
###compare with previous version
protein_age <- read.csv("/vol/projects/yzhang/500FG_aging/output/02_Olink/olink_age_linermodel_ynorm(FDR).csv",row.names=1)
head(protein_age)
dim(protein_age)
protein_age_sig <- protein_age[protein_age$sig=="sig",]
head(protein_age_sig)
dim(protein_age_sig)

,X,olink,estimate,p,padj,value,sig
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,Estimate,IL8,0.0244614877,1.974530e-08,1.727713e-07,1.884644e-01,sig
2,Estimate1,VEGFA,0.0035821211,6.823744e-03,1.492694e-02,7.758793e-03,sig
3,Estimate2,CD8A,-0.0196943936,9.880364e-06,4.610836e-05,-9.857491e-02,sig
4,Estimate3,CDCP1,0.1036950514,1.565948e-44,1.096164e-42,4.542385e+00,sig
5,Estimate4,CD244,0.0006495725,7.656722e-01,8.245701e-01,7.532253e-05,no
6,Estimate5,IL7,0.0165106485,3.746174e-02,5.959823e-02,2.355099e-02,no


[1] 70  7

,X,olink,estimate,p,padj,value,sig
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,Estimate,IL8,0.024461488,1.974530e-08,1.727713e-07,0.188464421,sig
2,Estimate1,VEGFA,0.003582121,6.823744e-03,1.492694e-02,0.007758793,sig
3,Estimate2,CD8A,-0.019694394,9.880364e-06,4.610836e-05,-0.098574912,sig
4,Estimate3,CDCP1,0.103695051,1.565948e-44,1.096164e-42,4.542384803,sig
7,Estimate6,OPG,0.005531198,3.684785e-03,9.211963e-03,0.013460657,sig
8,Estimate7,LAP_TGF_beta_1,0.007241308,1.580669e-02,2.984128e-02,0.013042749,sig


[1] 40  7

In [14]:
length(intersect(protein_sig$feature, protein_age_sig$olink))
setdiff(protein_sig$feature, protein_age_sig$olink) # in 1st but not in 2nd
setdiff(protein_age_sig$olink, protein_sig$feature)  # in 2nd but not in 1st

[1] 35

[1] "CXCL5"   "TNFRSF9"

[1] "uPA"    "TRAIL"  "MMP_10" "FGF_19" "TWEAK"

In [15]:
save.image("/vol/projects/yzhang/500FG_aging/output/02_Olink/olink_gender.RData")

In [2]:
load("/vol/projects/yzhang/500FG_aging/output/02_Olink/olink_gender.RData")

In [5]:
final_result_sig <- final_result %>%
  filter(sig == "sig") %>%
  select(feature, estimate, p, padj)
head(final_result_sig)
dim(final_result_sig)
write_excel_csv(final_result_sig, "/vol/projects/yzhang/500FG_aging/output/02_Olink/olink_age_liner_gender_FDR_sig.csv")

,feature,estimate,p,padj
,<chr>,<dbl>,<dbl>,<dbl>
Estimate,IL8,0.0018572263,1.605897e-07,1.405159e-06
Estimate1,VEGFA,0.0003326290,2.074781e-03,6.051445e-03
Estimate2,CD8A,-0.0017322309,1.947852e-06,1.136247e-05
Estimate3,CDCP1,0.0082331461,1.164648e-42,8.152538e-41
Estimate6,OPG,0.0005986621,5.789523e-05,2.383921e-04
Estimate7,LAP_TGF_beta_1,0.0005582286,2.338438e-02,4.546963e-02


[1] 37  4